# Information-preserving transformation of one AirfRANS simulation

`T_geom` (RBF morphing + radial wall correction) and `T_bal` (radial eCDF) on a single simulation, with the
Jacobian determinant of the morphing at every node and cell centroid. Set `DATA_DIR` to the unzipped AirfRANS
`Dataset` folder (or the environment variable `AIRFRANS_DATA_DIR`). Runs on CPU in about a minute.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from dgp.datasets import airfrans as A
from dgp.morphing import StrictECDF

DATA_DIR = os.environ.get("AIRFRANS_DATA_DIR", "/path/to/AirfRANS/Dataset")
name = A.manifest(DATA_DIR)["full_test"][0]
print(name)

## T_geom
The mesh is cut to the box [-2, 4] x [-1.5, 1.5] and aligned (leading edge at (0, 0), trailing edge at (1, 0)).
The airfoil is mapped onto the circle of radius 0.5 centred at (0.5, 0) and the box onto the concentric circle
through its corners; the interior follows the RBF displacement field.

In [ ]:
sample = A.load_sample(DATA_DIR, name, with_cells=True)
out = A.transform_geometry(sample, jacobian=True)
for k, v in out["diag"].items():
    print(f"{k:24s} {v}")

## T_bal
In the datasets the eCDF of the radial coordinate is pooled over the training meshes
(`scripts/transform_airfrans.py`); here, for illustration, it is estimated on this mesh only.

In [ ]:
ecdf = StrictECDF(A.R_WALL, A.R_OUT).update(out["r"]).finalize()
r_bal = ecdf(out["r"])
X_Tbal = np.column_stack([out["theta"], r_bal])     # (rotated polar angle, balanced radius)

In [ ]:
u = sample["fields"][:, 0]
idx = np.random.default_rng(0).choice(len(u), 60_000, replace=False)
panels = [(sample["X"], "physical domain", (-0.5, 2.0), (-0.6, 0.6)),
          (out["X_Tgeom"], "after $T_{geom}$", None, None),
          (X_Tbal, r"after $T_{bal} \circ T_{geom}$ ($\theta$, $\hat r$)", None, None)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (X, title, xl, yl) in zip(axes, panels):
    ax.scatter(X[idx, 0], X[idx, 1], c=u[idx], s=0.3, cmap="turbo")
    ax.set_title(title)
    if xl:
        ax.set_xlim(*xl)
        ax.set_ylim(*yl)
    if X is not X_Tbal:
        ax.set_aspect("equal")
plt.tight_layout()

## One-to-one check
det J = det(I + grad delta) of the RBF map at every node; it must be strictly positive (no folding).

In [ ]:
det = out["det"]
print(f"min det J at the nodes: {det.min():.4f} ({(det <= 0).sum()} folded nodes)")
print(f"min det J at the cell centroids: {out['diag']['min_det_centroids']:.4f}")
plt.figure()
plt.hist(det, bins=200, log=True)
plt.xlabel("det J")
plt.ylabel("nodes");